[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_12/2_sql.ipynb)

# Day 12: SQL (medium)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (31 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Month over month at the gift shop | medium | 6 |
| Q2 | Did the viewer like it right away? | medium | 7 |
| Q3 | How long until customers come back? | medium | 7 |
| Q4 | Customers who always spend more | medium | 6 |
| Q5 | Two best products in every category, 2022 (review) | medium | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up short-video product tables (users, videos, events): small, for exam-style questions.
db.executescript("""
CREATE TABLE users (
    user_id     INTEGER PRIMARY KEY,
    signup_date TEXT NOT NULL,      -- 'YYYY-MM-DD'
    country     TEXT,               -- NULL when unknown
    platform    TEXT NOT NULL       -- 'ios', 'android' or 'web'
);
CREATE TABLE videos (
    video_id     INTEGER PRIMARY KEY,
    creator_id   INTEGER NOT NULL REFERENCES users(user_id),
    upload_date  TEXT NOT NULL,     -- 'YYYY-MM-DD'
    category     TEXT NOT NULL,
    duration_sec INTEGER NOT NULL
);
CREATE TABLE events (
    event_id   INTEGER PRIMARY KEY,
    user_id    INTEGER NOT NULL REFERENCES users(user_id),
    video_id   INTEGER NOT NULL REFERENCES videos(video_id),
    event_type TEXT NOT NULL,       -- 'view', 'like', 'comment' or 'share'
    event_time TEXT NOT NULL,       -- 'YYYY-MM-DD HH:MM:SS'
    watch_sec  INTEGER              -- seconds watched; only for 'view', NULL otherwise
);
INSERT INTO users VALUES
(1, '2024-03-11', 'BR', 'web'),
(2, '2024-03-02', 'KR', 'ios'),
(3, '2024-03-12', 'US', 'android'),
(4, '2024-03-02', 'KR', 'web'),
(5, '2024-03-14', 'KR', 'android'),
(6, '2024-03-03', 'KR', 'ios'),
(7, '2024-03-19', NULL, 'android'),
(8, '2024-03-19', 'US', 'web'),
(9, '2024-03-02', 'ID', 'ios'),
(10, '2024-03-18', 'BR', 'android'),
(11, '2024-03-14', 'BR', 'ios'),
(12, '2024-03-19', 'DE', 'android'),
(13, '2024-03-04', 'ID', 'android'),
(14, '2024-03-04', 'KR', 'ios'),
(15, '2024-03-07', 'US', 'web'),
(16, '2024-03-11', NULL, 'web'),
(17, '2024-03-12', 'DE', 'android'),
(18, '2024-03-06', 'ID', 'ios'),
(19, '2024-03-19', 'DE', 'web'),
(20, '2024-03-11', 'US', 'android');
INSERT INTO videos VALUES
(101, 12, '2024-02-22', 'comedy', 90),
(102, 9, '2024-02-25', 'food', 30),
(103, 9, '2024-03-04', 'comedy', 120),
(104, 2, '2024-03-15', 'education', 90),
(105, 5, '2024-03-01', 'pets', 45),
(106, 12, '2024-03-06', 'education', 60),
(107, 2, '2024-02-22', 'food', 60),
(108, 14, '2024-03-12', 'comedy', 15),
(109, 14, '2024-03-13', 'food', 120),
(110, 12, '2024-03-12', 'gaming', 45),
(111, 14, '2024-03-03', 'pets', 45),
(112, 2, '2024-03-05', 'food', 30),
(113, 12, '2024-02-23', 'gaming', 15),
(114, 3, '2024-03-15', 'food', 30),
(115, 14, '2024-02-27', 'gaming', 60);
INSERT INTO events VALUES
(1, 9, 107, 'view', '2024-03-02 12:41:59', 4),
(2, 9, 107, 'like', '2024-03-02 12:42:16', NULL),
(3, 2, 107, 'view', '2024-03-02 14:06:14', 14),
(4, 2, 107, 'like', '2024-03-02 14:06:44', NULL),
(5, 9, 105, 'view', '2024-03-03 09:17:05', 28),
(6, 2, 101, 'view', '2024-03-03 09:53:42', 51),
(7, 2, 107, 'view', '2024-03-03 09:54:49', 52),
(8, 2, 107, 'comment', '2024-03-03 09:55:54', NULL),
(9, 6, 105, 'view', '2024-03-03 22:58:43', 4),
(10, 6, 105, 'share', '2024-03-03 22:58:52', NULL),
(11, 6, 101, 'view', '2024-03-04 11:00:30', 64),
(12, 6, 101, 'like', '2024-03-04 11:01:38', NULL),
(13, 6, 111, 'view', '2024-03-04 11:02:01', 35),
(14, 6, 111, 'like', '2024-03-04 11:02:51', NULL),
(15, 14, 105, 'view', '2024-03-04 15:34:40', 25),
(16, 14, 105, 'like', '2024-03-04 15:35:17', NULL),
(17, 13, 102, 'view', '2024-03-04 19:46:25', 2),
(18, 14, 101, 'view', '2024-03-05 12:39:47', 39),
(19, 6, 105, 'view', '2024-03-05 22:01:18', 6),
(20, 2, 112, 'view', '2024-03-06 07:09:37', 27),
(21, 6, 103, 'view', '2024-03-06 09:37:05', 97),
(22, 18, 103, 'view', '2024-03-06 21:13:43', 52),
(23, 18, 107, 'view', '2024-03-06 21:15:19', 42),
(24, 18, 107, 'like', '2024-03-06 21:16:10', NULL),
(25, 18, 107, 'share', '2024-03-06 21:16:13', NULL),
(26, 15, 101, 'view', '2024-03-07 11:44:01', 7),
(27, 15, 101, 'like', '2024-03-07 11:44:10', NULL),
(28, 15, 107, 'view', '2024-03-07 11:44:19', 14),
(29, 14, 106, 'view', '2024-03-08 08:14:09', 41),
(30, 13, 111, 'view', '2024-03-08 08:35:09', 7),
(31, 9, 112, 'view', '2024-03-08 13:23:34', 8),
(32, 9, 112, 'like', '2024-03-08 13:23:58', NULL),
(33, 9, 112, 'comment', '2024-03-08 13:24:12', NULL),
(34, 9, 111, 'view', '2024-03-08 13:24:57', 4),
(35, 6, 112, 'view', '2024-03-08 18:14:31', 14),
(36, 6, 112, 'like', '2024-03-08 18:14:46', NULL),
(37, 14, 101, 'view', '2024-03-09 08:01:03', 74),
(38, 14, 101, 'comment', '2024-03-09 08:02:29', NULL),
(39, 13, 102, 'view', '2024-03-09 12:33:10', 5),
(40, 13, 105, 'view', '2024-03-09 12:34:11', 21),
(41, 13, 105, 'like', '2024-03-09 12:34:34', NULL),
(42, 6, 107, 'view', '2024-03-09 20:22:24', 9),
(43, 6, 107, 'comment', '2024-03-09 20:22:44', NULL),
(44, 2, 103, 'view', '2024-03-10 10:33:47', 57),
(45, 16, 106, 'view', '2024-03-11 07:23:13', 18),
(46, 16, 105, 'view', '2024-03-11 07:24:16', 31),
(47, 16, 105, 'like', '2024-03-11 07:25:07', NULL),
(48, 1, 106, 'view', '2024-03-11 12:28:25', 58),
(49, 1, 106, 'like', '2024-03-11 12:29:37', NULL),
(50, 20, 101, 'view', '2024-03-11 12:34:38', 22),
(51, 20, 106, 'view', '2024-03-11 12:35:47', 55),
(52, 6, 102, 'view', '2024-03-11 16:16:23', 14),
(53, 6, 102, 'share', '2024-03-11 16:16:51', NULL),
(54, 18, 111, 'view', '2024-03-11 19:33:14', 31),
(55, 18, 111, 'like', '2024-03-11 19:33:50', NULL),
(56, 18, 111, 'share', '2024-03-11 19:33:57', NULL),
(57, 18, 105, 'view', '2024-03-11 19:34:32', 11),
(58, 20, 101, 'view', '2024-03-12 07:01:39', 89),
(59, 20, 111, 'view', '2024-03-12 07:03:19', 32),
(60, 20, 111, 'share', '2024-03-12 07:04:05', NULL),
(61, 16, 110, 'view', '2024-03-12 11:55:53', 44),
(62, 16, 106, 'view', '2024-03-12 11:57:26', 39),
(63, 16, 106, 'like', '2024-03-12 11:58:16', NULL),
(64, 14, 105, 'view', '2024-03-12 12:08:00', 11),
(65, 14, 105, 'comment', '2024-03-12 12:08:16', NULL),
(66, 14, 106, 'view', '2024-03-12 12:09:11', 27),
(67, 14, 106, 'share', '2024-03-12 12:09:56', NULL),
(68, 13, 112, 'view', '2024-03-12 14:39:25', 29),
(69, 13, 112, 'like', '2024-03-12 14:40:10', NULL),
(70, 6, 106, 'view', '2024-03-12 15:06:03', 42),
(71, 17, 105, 'view', '2024-03-12 16:27:17', 8),
(72, 17, 105, 'comment', '2024-03-12 16:27:32', NULL),
(73, 17, 101, 'view', '2024-03-12 16:28:06', 3),
(74, 3, 110, 'view', '2024-03-12 20:07:25', 22),
(75, 3, 110, 'like', '2024-03-12 20:07:55', NULL),
(76, 2, 103, 'view', '2024-03-13 15:34:26', 9),
(77, 2, 110, 'view', '2024-03-13 15:35:17', 28),
(78, 20, 110, 'view', '2024-03-13 18:21:30', 37),
(79, 6, 112, 'view', '2024-03-13 20:56:01', 14),
(80, 6, 102, 'view', '2024-03-13 20:57:06', 3),
(81, 13, 105, 'view', '2024-03-14 10:09:15', 4),
(82, 13, 107, 'view', '2024-03-14 10:10:06', 9),
(83, 1, 103, 'view', '2024-03-14 11:05:11', 31),
(84, 1, 103, 'comment', '2024-03-14 11:06:01', NULL),
(85, 15, 103, 'view', '2024-03-14 16:30:06', 14),
(86, 5, 103, 'view', '2024-03-14 17:46:31', 38),
(87, 5, 103, 'share', '2024-03-14 17:47:26', NULL),
(88, 5, 108, 'view', '2024-03-14 17:48:11', 13),
(89, 6, 107, 'view', '2024-03-15 11:10:30', 23),
(90, 6, 107, 'like', '2024-03-15 11:11:02', NULL),
(91, 3, 103, 'view', '2024-03-15 11:16:56', 61),
(92, 3, 103, 'like', '2024-03-15 11:18:01', NULL),
(93, 3, 104, 'view', '2024-03-15 11:18:48', 22),
(94, 2, 114, 'view', '2024-03-15 12:38:00', 27),
(95, 2, 114, 'like', '2024-03-15 12:38:32', NULL),
(96, 2, 101, 'view', '2024-03-15 12:39:12', 43),
(97, 20, 108, 'view', '2024-03-15 16:22:52', 8),
(98, 20, 108, 'like', '2024-03-15 16:23:09', NULL),
(99, 20, 111, 'view', '2024-03-15 16:23:27', 33),
(100, 1, 106, 'view', '2024-03-15 18:39:36', 10),
(101, 1, 112, 'view', '2024-03-15 18:40:34', 3),
(102, 17, 104, 'view', '2024-03-15 19:00:47', 57),
(103, 17, 104, 'view', '2024-03-15 19:02:43', 87),
(104, 20, 109, 'view', '2024-03-16 08:25:46', 115),
(105, 3, 101, 'view', '2024-03-16 09:46:23', 45),
(106, 3, 101, 'share', '2024-03-16 09:47:19', NULL),
(107, 3, 110, 'view', '2024-03-16 09:47:57', 20),
(108, 3, 110, 'comment', '2024-03-16 09:48:25', NULL),
(109, 2, 101, 'view', '2024-03-16 14:12:17', 14),
(110, 15, 114, 'view', '2024-03-16 16:03:45', 13),
(111, 15, 105, 'view', '2024-03-16 16:04:33', 41),
(112, 15, 105, 'share', '2024-03-16 16:05:31', NULL),
(113, 15, 104, 'view', '2024-03-17 08:34:36', 13),
(114, 15, 104, 'view', '2024-03-17 08:35:27', 38),
(115, 15, 104, 'share', '2024-03-17 08:36:21', NULL),
(116, 3, 103, 'view', '2024-03-17 08:57:49', 36),
(117, 3, 105, 'view', '2024-03-17 08:59:13', 27),
(118, 3, 105, 'like', '2024-03-17 08:59:57', NULL),
(119, 1, 111, 'view', '2024-03-17 19:06:30', 27),
(120, 1, 111, 'like', '2024-03-17 19:07:00', NULL),
(121, 6, 111, 'view', '2024-03-17 19:07:10', 12),
(122, 1, 106, 'view', '2024-03-17 19:07:12', 40),
(123, 6, 111, 'like', '2024-03-17 19:07:39', NULL),
(124, 1, 106, 'like', '2024-03-17 19:07:53', NULL),
(125, 6, 108, 'view', '2024-03-17 19:07:58', 7),
(126, 20, 111, 'view', '2024-03-17 22:38:49', 5),
(127, 14, 101, 'view', '2024-03-18 08:03:34', 53),
(128, 14, 101, 'like', '2024-03-18 08:04:33', NULL),
(129, 14, 101, 'comment', '2024-03-18 08:04:37', NULL),
(130, 14, 101, 'share', '2024-03-18 08:04:55', NULL),
(131, 10, 112, 'view', '2024-03-18 10:26:31', 16),
(132, 10, 112, 'like', '2024-03-18 10:26:52', NULL),
(133, 20, 111, 'view', '2024-03-18 13:02:42', 31),
(134, 6, 106, 'view', '2024-03-18 17:35:05', 17),
(135, 3, 105, 'view', '2024-03-18 20:57:04', 3),
(136, 3, 105, 'share', '2024-03-18 20:57:15', NULL),
(137, 20, 103, 'view', '2024-03-19 07:23:55', 102),
(138, 20, 103, 'like', '2024-03-19 07:25:46', NULL),
(139, 7, 105, 'view', '2024-03-19 08:41:45', 10),
(140, 18, 103, 'view', '2024-03-19 15:27:43', 63),
(141, 18, 103, 'like', '2024-03-19 15:28:55', NULL),
(142, 18, 108, 'view', '2024-03-19 15:29:20', 9),
(143, 12, 105, 'view', '2024-03-19 17:04:25', 17),
(144, 17, 112, 'view', '2024-03-19 17:16:40', 5),
(145, 17, 111, 'view', '2024-03-19 17:17:35', 12),
(146, 17, 111, 'like', '2024-03-19 17:18:01', NULL),
(147, 17, 111, 'share', '2024-03-19 17:18:15', NULL),
(148, 1, 105, 'view', '2024-03-19 19:09:40', 24),
(149, 3, 109, 'view', '2024-03-19 21:00:21', 55),
(150, 8, 114, 'view', '2024-03-19 22:12:19', 28),
(151, 8, 114, 'like', '2024-03-19 22:13:02', NULL),
(152, 9, 102, 'view', '2024-03-20 07:52:14', 17),
(153, 7, 104, 'view', '2024-03-20 16:33:37', 51),
(154, 7, 104, 'like', '2024-03-20 16:34:48', NULL),
(155, 7, 104, 'comment', '2024-03-20 16:35:06', NULL),
(156, 18, 110, 'view', '2024-03-20 19:03:05', 22),
(157, 18, 110, 'view', '2024-03-20 19:04:12', 2),
(158, 18, 110, 'share', '2024-03-20 19:04:24', NULL),
(159, 14, 103, 'view', '2024-03-20 20:52:39', 67),
(160, 14, 103, 'like', '2024-03-20 20:53:56', NULL),
(161, 12, 104, 'view', '2024-03-21 22:56:52', 59),
(162, 12, 104, 'comment', '2024-03-21 22:58:01', NULL),
(163, 12, 103, 'view', '2024-03-22 09:23:32', 59),
(164, 13, 114, 'view', '2024-03-22 21:15:28', 21),
(165, 13, 107, 'view', '2024-03-22 21:16:24', 8),
(166, 13, 107, 'like', '2024-03-22 21:16:44', NULL),
(167, 13, 107, 'share', '2024-03-22 21:16:59', NULL),
(168, 6, 110, 'view', '2024-03-23 08:31:17', 25),
(169, 6, 110, 'like', '2024-03-23 08:31:59', NULL),
(170, 6, 104, 'view', '2024-03-23 08:32:58', 13),
(171, 6, 104, 'like', '2024-03-23 08:33:19', NULL),
(172, 12, 106, 'view', '2024-03-23 13:02:23', 11),
(173, 12, 106, 'like', '2024-03-23 13:02:43', NULL),
(174, 12, 106, 'comment', '2024-03-23 13:02:50', NULL),
(175, 12, 106, 'view', '2024-03-23 13:03:47', 28),
(176, 3, 111, 'view', '2024-03-23 13:59:19', 21),
(177, 1, 108, 'view', '2024-03-23 17:47:16', 15),
(178, 14, 111, 'view', '2024-03-23 21:05:47', 30),
(179, 14, 111, 'like', '2024-03-23 21:06:21', NULL),
(180, 14, 106, 'view', '2024-03-23 21:06:33', 59),
(181, 3, 101, 'view', '2024-03-24 07:16:02', 4),
(182, 3, 104, 'view', '2024-03-24 07:16:41', 59),
(183, 3, 104, 'like', '2024-03-24 07:17:54', NULL),
(184, 13, 106, 'view', '2024-03-24 09:59:46', 51),
(185, 13, 106, 'comment', '2024-03-24 10:00:54', NULL),
(186, 20, 107, 'view', '2024-03-24 10:49:51', 38),
(187, 20, 107, 'share', '2024-03-24 10:50:42', NULL),
(188, 14, 104, 'view', '2024-03-24 15:18:41', 12),
(189, 14, 104, 'comment', '2024-03-24 15:19:02', NULL),
(190, 14, 112, 'view', '2024-03-24 15:20:00', 8),
(191, 1, 102, 'view', '2024-03-24 16:41:55', 24),
(192, 5, 108, 'view', '2024-03-24 18:06:24', 10),
(193, 5, 108, 'like', '2024-03-24 18:06:35', NULL),
(194, 18, 104, 'view', '2024-03-24 18:50:09', 53),
(195, 18, 114, 'view', '2024-03-24 18:51:45', 4),
(196, 12, 102, 'view', '2024-03-24 22:04:26', 27),
(197, 12, 111, 'view', '2024-03-24 22:05:03', 12),
(198, 9, 106, 'view', '2024-03-25 11:38:15', 57),
(199, 9, 106, 'like', '2024-03-25 11:39:24', NULL),
(200, 9, 104, 'view', '2024-03-25 11:40:01', 52),
(201, 9, 104, 'share', '2024-03-25 11:40:55', NULL),
(202, 14, 111, 'view', '2024-03-25 19:58:54', 44),
(203, 13, 108, 'view', '2024-03-26 10:12:08', 6),
(204, 3, 112, 'view', '2024-03-26 14:21:12', 25),
(205, 8, 101, 'view', '2024-03-26 19:03:13', 78),
(206, 8, 101, 'like', '2024-03-26 19:04:33', NULL),
(207, 5, 108, 'view', '2024-03-27 09:47:47', 6),
(208, 7, 104, 'view', '2024-03-27 16:03:01', 65),
(209, 7, 104, 'share', '2024-03-27 16:04:14', NULL),
(210, 7, 107, 'view', '2024-03-27 16:05:01', 25),
(211, 7, 107, 'like', '2024-03-27 16:05:28', NULL),
(212, 6, 114, 'view', '2024-03-27 20:45:48', 17),
(213, 6, 114, 'share', '2024-03-27 20:46:22', NULL),
(214, 18, 102, 'view', '2024-03-27 21:42:56', 19),
(215, 18, 102, 'like', '2024-03-27 21:43:29', NULL),
(216, 17, 104, 'view', '2024-03-28 12:45:50', 68),
(217, 17, 104, 'like', '2024-03-28 12:47:17', NULL),
(218, 1, 107, 'view', '2024-03-28 13:51:15', 49),
(219, 13, 110, 'view', '2024-03-28 15:10:20', 19),
(220, 13, 104, 'view', '2024-03-28 15:11:14', 77),
(221, 13, 104, 'like', '2024-03-28 15:12:48', NULL),
(222, 3, 102, 'view', '2024-03-28 20:10:03', 23),
(223, 3, 110, 'view', '2024-03-28 20:10:49', 17),
(224, 3, 110, 'comment', '2024-03-28 20:11:12', NULL),
(225, 1, 104, 'view', '2024-03-29 15:30:16', 90),
(226, 13, 107, 'view', '2024-03-29 15:43:20', 12),
(227, 3, 104, 'view', '2024-03-29 17:35:20', 6),
(228, 3, 106, 'view', '2024-03-29 17:35:31', 26),
(229, 3, 106, 'like', '2024-03-29 17:36:06', NULL),
(230, 20, 111, 'view', '2024-03-30 13:57:09', 2),
(231, 20, 111, 'comment', '2024-03-30 13:57:15', NULL),
(232, 20, 102, 'view', '2024-03-30 13:58:14', 8),
(233, 20, 102, 'share', '2024-03-30 13:58:41', NULL),
(234, 15, 109, 'view', '2024-03-30 22:50:44', 102),
(235, 15, 109, 'like', '2024-03-30 22:52:37', NULL),
(236, 6, 112, 'view', '2024-03-31 10:52:46', 22),
(237, 6, 112, 'share', '2024-03-31 10:53:10', NULL),
(238, 3, 101, 'view', '2024-03-31 11:25:37', 52),
(239, 3, 101, 'like', '2024-03-31 11:26:39', NULL),
(240, 3, 101, 'share', '2024-03-31 11:26:56', NULL),
(241, 14, 102, 'view', '2024-03-31 19:39:37', 20),
(242, 14, 102, 'share', '2024-03-31 19:40:01', NULL);
""")

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: Northwind orders (enlarged version, 2012 to 2023). Copied into tables with an nw_ prefix.
db.execute("ATTACH DATABASE ? AS nw", (data_path("northwind.db", "https://raw.githubusercontent.com/jpwhite3/northwind-SQLite3/main/dist/northwind.db"),))
db.executescript("""
CREATE TABLE nw_orders AS SELECT OrderID AS order_id, CustomerID AS customer_id, EmployeeID AS employee_id,
       substr(OrderDate, 1, 10) AS order_date, ShipCountry AS ship_country FROM nw.Orders;
CREATE TABLE nw_order_details AS SELECT OrderID AS order_id, ProductID AS product_id, UnitPrice AS unit_price,
       Quantity AS quantity, Discount AS discount FROM nw."Order Details";
CREATE TABLE nw_products AS SELECT ProductID AS product_id, ProductName AS product_name, CategoryID AS category_id
       FROM nw.Products;
CREATE TABLE nw_categories AS SELECT CategoryID AS category_id, CategoryName AS category_name FROM nw.Categories;
""")
db.execute("DETACH DATABASE nw")

# Real data: UCI Online Retail (UK online shop, 2010-12 to 2011-12; CC BY 4.0).
def online_retail():
    path = os.path.join(DATA, "online_retail.csv.gz")
    if not os.path.exists(path):
        import io, zipfile
        print("downloading Online Retail (about 23 MB, then converting the xlsx)")
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://archive.ics.uci.edu/static/public/352/online+retail.zip").read()))
        pd.read_excel(io.BytesIO(z.read("Online Retail.xlsx"))).to_csv(path, index=False, compression="gzip")
    return pd.read_csv(path, dtype={"InvoiceNo": str, "StockCode": str})

_o = online_retail()
_o.columns = ["invoice_no", "stock_code", "description", "quantity", "invoice_date", "unit_price", "customer_id",
              "country"]
_o["customer_id"] = _o["customer_id"].astype("Int64")
_o["invoice_date"] = pd.to_datetime(_o["invoice_date"]).dt.strftime("%Y-%m-%d %H:%M:%S")
_o.to_sql("retail", db, index=False)

print("tables:", ", ".join(r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

## Tables

**Made-up product tables** (a small short-video app, invented for practice):

| Table | One row is | Columns |
|---|---|---|
| `users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (can be NULL), `platform` (ios / android / web) |
| `videos` | one uploaded video | `video_id`, `creator_id` (a user_id), `upload_date`, `category`, `duration_sec` |
| `events` | one user action on a video | `event_id`, `user_id`, `video_id`, `event_type` (view / like / comment / share), `event_time` ('YYYY-MM-DD HH:MM:SS'), `watch_sec` (views only, else NULL) |

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real tables: Northwind** (orders of a food wholesaler; enlarged build with dates 2012 to 2023; MIT license,
jpwhite3/northwind-SQLite3). Columns renamed to snake_case:

| Table | One row is | Columns |
|---|---|---|
| `nw_orders` | one order (16,282) | `order_id`, `customer_id`, `employee_id`, `order_date` ('YYYY-MM-DD'), `ship_country` |
| `nw_order_details` | one product line in an order (609,283) | `order_id`, `product_id`, `unit_price`, `quantity`, `discount` (0 to 1). Revenue = `unit_price * quantity * (1 - discount)` |
| `nw_products` / `nw_categories` | one product (77) / category (8) | `product_id`, `product_name`, `category_id` / `category_id`, `category_name` |

**Real table: UCI Online Retail** (a UK online gift shop, 2010-12-01 to 2011-12-09; CC BY 4.0):

| Table | One row is | Columns |
|---|---|---|
| `retail` | one product line of an invoice (541,909) | `invoice_no` (starts with 'C' = cancellation), `stock_code`, `description`, `quantity` (negative for returns), `invoice_date` ('YYYY-MM-DD HH:MM:SS'), `unit_price`, `customer_id` (NULL for about 25% of rows), `country` |

The queries run in SQLite. They are written in standard SQL that also works in PostgreSQL; differences (dates, integer division) are noted in the solutions.

---
### Q1. Month over month at the gift shop

*medium, about 6 min*

Online Retail (real). Net revenue of a line is `quantity * unit_price` (cancellations have negative
quantity, so they subtract). For every month, return `month` ('YYYY-MM'), `revenue`, `prev_revenue`, `mom_pct`
(percent change versus the previous month, 1 decimal) and `next_pct` (percent change from this month to the next one).
Round money to whole units.

Example of the logic: if November is 1,461,756 and December is 433,668, December's `mom_pct` is -70.3.

Then answer: the CEO sees `-70.3` for December 2011 and panics. What do you check before anybody panics?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "compare a row with the previous / next row". Pattern: LAG and LEAD over `ORDER BY month` on a monthly aggregate.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: `strftime('%Y-%m', invoice_date)` and SUM(quantity * unit_price) per month.
2. `LAG(revenue) OVER (ORDER BY month)` for the previous month, LEAD for the next one.
3. Percent change = `100.0 * (revenue - prev) / prev`. The first month has no previous value (NULL).

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH monthly AS (
    SELECT strftime('%Y-%m', invoice_date) AS month,
           SUM(quantity * unit_price)     AS revenue
    FROM retail
    GROUP BY strftime('%Y-%m', invoice_date)
)
SELECT month,
       ROUND(revenue, 0)                              AS revenue,
       ROUND(LAG(revenue) OVER (ORDER BY month), 0)   AS prev_revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1) AS mom_pct,
       ROUND(100.0 * (LEAD(revenue) OVER (ORDER BY month) - revenue) / revenue, 1) AS next_pct
FROM monthly
ORDER BY month
```

Result:

| month | revenue | prev_revenue | mom_pct | next_pct |
|---|---|---|---|---|
| 2010-12 | 748957.0 | NULL | NULL | -25.2 |
| 2011-01 | 560000.0 | 748957.0 | -25.2 | -11.1 |
| 2011-02 | 498063.0 | 560000.0 | -11.1 | 37.2 |
| 2011-03 | 683267.0 | 498063.0 | 37.2 | -27.8 |
| 2011-04 | 493207.0 | 683267.0 | -27.8 | 46.7 |
| 2011-05 | 723334.0 | 493207.0 | 46.7 | -4.5 |
| 2011-06 | 691123.0 | 723334.0 | -4.5 | -1.4 |
| 2011-07 | 681300.0 | 691123.0 | -1.4 | 0.2 |
| 2011-08 | 682681.0 | 681300.0 | 0.2 | 49.4 |
| 2011-09 | 1019688.0 | 682681.0 | 49.4 | 5.0 |
| 2011-10 | 1070705.0 | 1019688.0 | 5.0 | 36.5 |
| 2011-11 | 1461756.0 | 1070705.0 | 36.5 | -70.3 |

(13 rows, first 12 shown)

**Why:** LAG reads a value from the previous row of the window order, LEAD from the next row, without a self join. The first month has no previous row, so its mom_pct is NULL (and the last month has no next_pct). December 2011: the data ends on 2011-12-09, so the month is only 9 days long. Always check the date range before comparing periods: compare equal windows (Dec 1 to 9 versus Nov 1 to 9), or revenue per day, or the same month last year. November is also a seasonal peak (Christmas orders). PostgreSQL: `to_char(invoice_date, 'YYYY-MM')` or `date_trunc('month', invoice_date)`.

**Complexity:** One group by over 541,909 lines, then a window over 13 rows.

**Common mistakes:** Integer division when computing the percentage (use 100.0). Ordering the window by revenue instead of month. Forgetting that a missing month (no rows) makes LAG jump two months back: build a calendar of months if months can be empty. Panicking about a partial period.

**Learn more:** [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q2. Did the viewer like it right away?

*medium, about 7 min*

Short-video tables (made up). A **quick like** is a view whose **very next event by the same user**
is a `like` on the **same video**, at most **60 seconds** later.

For each video `category`, return `views`, `quick_likes` and `quick_like_rate` (3 decimals), highest rate first.

Example: user 9 views video 107 at 12:41:59 and likes video 107 at 12:42:16 (17 s later), so that view counts as a
quick like. If the user's next event were a view of another video, it would not count, even if a like came later.

Follow-up: two events of the same user can have the same `event_time`. How do you make "next event" well defined?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "the next event of the same user". Pattern: LEAD over `PARTITION BY user_id ORDER BY event_time`, then a conditional count.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: join events to videos; add `LEAD(event_type)`, `LEAD(video_id)` and `LEAD(event_time)` over `(PARTITION BY e.user_id ORDER BY e.event_time, e.event_id)`.
2. Keep rows with event_type = 'view'.
3. `SUM(CASE WHEN next_type = 'like' AND next_video = video_id AND seconds <= 60 THEN 1 ELSE 0 END)`; seconds in SQLite: `(julianday(t2) - julianday(t1)) * 86400`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH seq AS (
    SELECT e.event_id, e.user_id, e.video_id, e.event_type, e.event_time, v.category,
           LEAD(e.event_type) OVER w AS next_type,
           LEAD(e.video_id)   OVER w AS next_video,
           LEAD(e.event_time) OVER w AS next_time
    FROM events e
    JOIN videos v ON v.video_id = e.video_id
    WINDOW w AS (PARTITION BY e.user_id ORDER BY e.event_time, e.event_id)
), flagged AS (
    SELECT category,
           CASE WHEN next_type = 'like' AND next_video = video_id
                 AND (julianday(next_time) - julianday(event_time)) * 86400 <= 60
                THEN 1 ELSE 0 END AS quick_like
    FROM seq
    WHERE event_type = 'view'
)
SELECT category,
       COUNT(*) AS views,
       SUM(quick_like) AS quick_likes,
       ROUND(1.0 * SUM(quick_like) / COUNT(*), 3) AS quick_like_rate
FROM flagged
GROUP BY category
ORDER BY quick_like_rate DESC
```

Result:

| category | views | quick_likes | quick_like_rate |
|---|---|---|---|
| pets | 32 | 10 | 0.313 |
| food | 44 | 13 | 0.295 |
| gaming | 10 | 2 | 0.2 |
| education | 33 | 5 | 0.152 |
| comedy | 36 | 4 | 0.111 |

**Why:** LEAD looks one row ahead inside the user's own timeline, so "next event" is exactly what the question means; a self join with `MIN(event_time) > t` would also work but is longer and slower. The LEAD must be computed before filtering to views: if you filter first, the like rows are gone and LEAD sees the next view instead. Adding `event_id` to the ORDER BY breaks ties when two events share a timestamp, so the result is deterministic. The named `WINDOW w` clause works in SQLite and PostgreSQL. PostgreSQL seconds: `EXTRACT(EPOCH FROM next_time - event_time)`.

**Complexity:** One sort of events per user for the window, then one group by.

**Common mistakes:** Filtering `WHERE event_type = 'view'` in the same SELECT as the LEAD (WHERE runs first). Forgetting `PARTITION BY user_id`, which compares with another user's event. Not checking that the like is on the same video. Integer division in the rate.

**Learn more:** [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead), [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
### Q3. How long until customers come back?

*medium, about 7 min*

Online Retail (real). The marketing team wants to time its reminder emails. For each customer, look at
the **days on which they bought** (ignore rows without `customer_id` and cancellation invoices, whose number starts
with 'C'). The **gap** is the number of days between two consecutive buying days of the same customer.

Per `country`, return `repeat_customers` (customers with at least one gap), `n_gaps`, `avg_gap_days` (1 decimal)
and `min_gap`. Keep countries with at least 10 repeat customers, shortest average gap first.

Example: a customer who bought on 2011-01-10, 2011-01-10 (two invoices) and 2011-02-09 has one gap of 30 days, not
two gaps (0 and 30).

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "days between consecutive purchases of the same customer". Pattern: LAG over `PARTITION BY customer ORDER BY day`, after removing duplicates per day.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: `SELECT DISTINCT customer_id, country, substr(invoice_date, 1, 10) AS buy_day` with the filters.
2. Gap = `julianday(buy_day) - julianday(LAG(buy_day) OVER (PARTITION BY customer_id ORDER BY buy_day))`.
3. Outer query: drop NULL gaps (first purchase), group by country, HAVING COUNT(DISTINCT customer_id) >= 10.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH buy_days AS (
    SELECT DISTINCT customer_id, country, substr(invoice_date, 1, 10) AS buy_day
    FROM retail
    WHERE customer_id IS NOT NULL
      AND invoice_no NOT LIKE 'C%'
), gaps AS (
    SELECT country, customer_id, buy_day,
           julianday(buy_day)
             - julianday(LAG(buy_day) OVER (PARTITION BY customer_id ORDER BY buy_day)) AS gap_days
    FROM buy_days
)
SELECT country,
       COUNT(DISTINCT customer_id) AS repeat_customers,
       COUNT(*)                    AS n_gaps,
       ROUND(AVG(gap_days), 1)     AS avg_gap_days,
       MIN(gap_days)               AS min_gap
FROM gaps
WHERE gap_days IS NOT NULL
GROUP BY country
HAVING COUNT(DISTINCT customer_id) >= 10
ORDER BY avg_gap_days
```

Result:

| country | repeat_customers | n_gaps | avg_gap_days | min_gap |
|---|---|---|---|---|
| Germany | 67 | 323 | 42.2 | 1.0 |
| France | 59 | 275 | 43.6 | 1.0 |
| United Kingdom | 2522 | 11174 | 46.1 | 1.0 |
| Belgium | 18 | 71 | 48.9 | 2.0 |
| Portugal | 12 | 33 | 57.9 | 1.0 |
| Switzerland | 14 | 29 | 66.2 | 1.0 |
| Spain | 20 | 61 | 69.0 | 3.0 |

**Why:** DISTINCT per day first, otherwise several invoices on the same day create gaps of 0 that pull the average down. LAG gives the previous buying day of the same customer; the first day of each customer has no previous day (NULL) and is dropped. Germany and France come back a bit faster than the United Kingdom on average (about 42 to 46 days), so a reminder around day 35 to 40 is a reasonable first test. The median gap would be more robust than the mean (gaps are skewed). PostgreSQL: dates subtract directly, `buy_day - LAG(buy_day) OVER (...)` gives an integer number of days.

**Complexity:** DISTINCT and a window sort over about 400k purchase lines, then a small group by.

**Common mistakes:** Partitioning by country instead of customer (mixes customers). Forgetting the DISTINCT per day. Keeping the NULL first gaps in COUNT(*). A few customers appear with two countries; partitioning by customer_id still gives correct gaps, but they are then counted in both countries.

**Learn more:** [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates), [sql-dedup-cleaning](https://mahsa-agz.github.io/ds-handbook/#sql-dedup-cleaning)

</details>

---
### Q4. Customers who always spend more

*medium, about 6 min*

Shopping app (made up). Find users with **at least 3 orders** (any status) where **every order is
larger than the order before it** (by `order_time`). Return `user_id`, `n_orders`, `first_amount`, `last_amount`,
users with more orders first, then by user_id.

Example: amounts 10.50, 18.00, 24.56 qualify; 10.50, 18.00, 18.00 do not (equal is not larger); 30, 12, 40 do not.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "each row compared with the previous row of the same user". Pattern: LAG per user, then a group-level check that no row breaks the rule (count of violations = 0).

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: `LAG(amount) OVER (PARTITION BY user_id ORDER BY order_time, order_id)` and `COUNT(*) OVER (PARTITION BY user_id)`.
2. Group by user; HAVING the number of rows with `prev_amount IS NOT NULL AND amount <= prev_amount` is 0.
3. first and last amount: MIN and MAX work here because the sequence is increasing.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH p AS (
    SELECT user_id, order_id, order_time, amount,
           LAG(amount) OVER (PARTITION BY user_id ORDER BY order_time, order_id) AS prev_amount,
           COUNT(*)    OVER (PARTITION BY user_id)                                AS n_orders
    FROM purchases
)
SELECT user_id,
       n_orders,
       ROUND(MIN(amount), 2) AS first_amount,
       ROUND(MAX(amount), 2) AS last_amount
FROM p
WHERE n_orders >= 3
GROUP BY user_id, n_orders
HAVING SUM(CASE WHEN prev_amount IS NOT NULL AND amount <= prev_amount THEN 1 ELSE 0 END) = 0
ORDER BY n_orders DESC, user_id
```

Result:

| user_id | n_orders | first_amount | last_amount |
|---|---|---|---|
| 498 | 4 | 17.35 | 47.0 |
| 405 | 3 | 14.32 | 37.09 |
| 536 | 3 | 10.5 | 24.56 |
| 754 | 3 | 13.88 | 61.04 |
| 846 | 3 | 8.43 | 16.57 |

**Why:** "All rows satisfy X" is easiest as "no row violates X": count the violations with a CASE inside SUM and require 0. LAG gives each order its previous amount; the first order has NULL and is not a violation. The COUNT window adds the order count to every row without collapsing them. Because the amounts are strictly increasing for the users that pass, MIN is the first and MAX the last amount; in general you would use `FIRST_VALUE` / `LAST_VALUE` (with a full frame) instead.

**Complexity:** One window sort of purchases per user, one group by.

**Common mistakes:** Comparing with `<` instead of `<=` (equal amounts would pass). Ordering the window by amount instead of time. Using `LAST_VALUE(amount) OVER (ORDER BY ...)` with the default frame, which stops at the current row and returns the current amount.

**Learn more:** [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead), [sql-case-when](https://mahsa-agz.github.io/ds-handbook/#sql-case-when)

</details>

---
### Q5. Two best products in every category, 2022 (review)

*medium, about 5 min*

Northwind (real schema, enlarged dates). For orders placed in **2022**, return the **2 products with
the highest revenue in each category**. Revenue of a line is `unit_price * quantity * (1 - discount)`.
Keep ties (two products with exactly the same revenue both stay).

Return `category_name`, `dr`, `product_name`, `revenue` (rounded to whole units), ordered by category and dr.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: top 2 per category, ties kept. Pattern: aggregate, DENSE_RANK per category, filter in an outer query.

</details>

<details><summary><b>Hint 2</b></summary>

1. Join nw_order_details, nw_orders (for the date), nw_products, nw_categories.
2. Filter `order_date >= '2022-01-01' AND order_date < '2023-01-01'`, group by category and product.
3. `DENSE_RANK() OVER (PARTITION BY category_name ORDER BY revenue DESC)`, keep dr <= 2.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH rev AS (
    SELECT c.category_name, p.product_name,
           SUM(d.unit_price * d.quantity * (1 - d.discount)) AS revenue
    FROM nw_order_details d
    JOIN nw_orders o     ON o.order_id = d.order_id
    JOIN nw_products p   ON p.product_id = d.product_id
    JOIN nw_categories c ON c.category_id = p.category_id
    WHERE o.order_date >= '2022-01-01' AND o.order_date < '2023-01-01'
    GROUP BY c.category_name, p.product_name
), ranked AS (
    SELECT category_name, product_name, revenue,
           DENSE_RANK() OVER (PARTITION BY category_name ORDER BY revenue DESC) AS dr
    FROM rev
)
SELECT category_name, dr, product_name, ROUND(revenue, 0) AS revenue
FROM ranked
WHERE dr <= 2
ORDER BY category_name, dr
```

Result:

| category_name | dr | product_name | revenue |
|---|---|---|---|
| Beverages | 1 | Côte de Blaye | 4761445.0 |
| Beverages | 2 | Ipoh Coffee | 847136.0 |
| Condiments | 1 | Vegie-spread | 799639.0 |
| Condiments | 2 | Northwoods Cranberry Sauce | 686480.0 |
| Confections | 1 | Sir Rodney's Marmalade | 1469907.0 |
| Confections | 2 | Tarte au sucre | 867877.0 |
| Dairy Products | 1 | Raclette Courdavault | 1005290.0 |
| Dairy Products | 2 | Queso Manchego La Pastora | 706724.0 |
| Grains/Cereals | 1 | Gnocchi di nonna Alice | 659376.0 |
| Grains/Cereals | 2 | Wimmers gute Semmelknödel | 571268.0 |
| Meat/Poultry | 1 | Thüringer Rostbratwurst | 2139091.0 |
| Meat/Poultry | 2 | Mishi Kobe Niku | 1729219.0 |

(16 rows, first 12 shown)

**Why:** Same template as day 11: aggregate, rank inside the partition, filter outside. The date filter goes into WHERE before the GROUP BY, so only 2022 lines are summed. DENSE_RANK keeps ties; with ROW_NUMBER you would need a tie breaker.

**Complexity:** A join over the order lines of one year, a group by, one small window.

**Common mistakes:** Ranking by unit price or quantity instead of revenue. Forgetting the discount. Filtering on the year with `strftime('%Y', order_date) = '2022'` works but cannot use an index on order_date.

**Learn more:** [sql-top-n](https://mahsa-agz.github.io/ds-handbook/#sql-top-n), [sql-joins](https://mahsa-agz.github.io/ds-handbook/#sql-joins)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_12/3_stats.ipynb)